In [6]:
import pandas as pd
import numpy as np

#!pip install openpyxl

# chnage from scientific notation 
pd.set_option('display.float_format', lambda x: '%.5f' % x)

data = pd.read_csv("premier_league_top_scorers_full.csv") # upload csv

In [7]:
data.columns = data.columns.str.lower().str.replace(" ","_")
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 376 entries, 0 to 375
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   rank         376 non-null    int64
 1   player       376 non-null    str  
 2   club         376 non-null    str  
 3   goals        376 non-null    int64
 4   nationality  376 non-null    str  
 5   season       376 non-null    str  
dtypes: int64(2), str(4)
memory usage: 17.8 KB


In [10]:
data.head(5).to_string()

'   rank            player                 club  goals nationality   season\n0     1  Teddy Sheringham    Tottenham Hotspur     22     England  1992/93\n1     2     Les Ferdinand  Queens Park Rangers     20     England  1992/93\n2     3   Dean Holdsworth       Wimbledon F.C.     19     England  1992/93\n3     4       Micky Quinn        Coventry City     17     England  1992/93\n4     5      Alan Shearer     Blackburn Rovers     16     England  1992/93'

In [12]:
# Make sure goals and rank are numeric
data["rank"] = pd.to_numeric(data["rank"], errors="coerce")
data["goals"] = pd.to_numeric(data["goals"], errors="coerce")

In [33]:
player_goals = (
    data.groupby("player", as_index=False)
      .agg(
          total_goals=("goals", "sum"),
          appearances=("season", "count"),
          best_season_goals=("goals", "max")
      )
      .sort_values(["total_goals", "appearances"], ascending=[False, False])
)

player_goals

,player,total_goals,appearances,best_season_goals
0,Alan Shearer,236,10,34
78,Harry Kane,210,9,30
127,Mohamed Salah,184,8,32
171,Thierry Henry,164,7,30
164,Sergio Agüero,152,7,26
...,...,...,...,...
168,Steve Claridge,12,1,12
1,Alan Smith,11,1,11
66,Frédéric Kanouté,11,1,11
75,Gustavo Poyet,11,1,11


In [34]:
# Find each player's best goal total and the season it happened in
best_season = (
    data.sort_values(["player", "goals", "season"], ascending=[True, False, True])
        .groupby("player", as_index=False)
        .first()[["player", "season", "goals"]]
        .rename(columns={
            "season": "best_season",
            "goals": "best_season_goals"
        })  

)

best_season.sort_values(["best_season_goals", "best_season"], ascending=[False, True ]).head(10)


,player,best_season,best_season_goals
60,Erling Haaland,2022/23,36
9,Andy Cole,1993/94,34
0,Alan Shearer,1994/95,34
127,Mohamed Salah,2017/18,32
30,Cristiano Ronaldo,2007/08,31
109,Luis Suárez,2013/14,31
106,Kevin Phillips,1999/2000,30
171,Thierry Henry,2003/04,30
156,Robin van Persie,2011/12,30
78,Harry Kane,2017/18,30


In [36]:
# join best season to player_scores table
player_goals = player_goals.merge(
    best_season[["player", "best_season"]],
    on="player",
    how="left"
)

player_goals

,player,total_goals,appearances,best_season_goals,best_season
0,Alan Shearer,236,10,34,1994/95
1,Harry Kane,210,9,30,2017/18
2,Mohamed Salah,184,8,32,2017/18
3,Thierry Henry,164,7,30,2003/04
4,Sergio Agüero,152,7,26,2014/15
...,...,...,...,...,...
179,Steve Claridge,12,1,12,1996/97
180,Alan Smith,11,1,11,2000/01
181,Frédéric Kanouté,11,1,11,2000/01
182,Gustavo Poyet,11,1,11,2000/01


In [39]:
# Top 10 rows only
top10 = data[data["rank"] <= 10].copy()

player_top10_count = (
    top10.groupby("player", as_index=False)
        .agg(
            top10_appearances=("season", "count"),
            top10_goals=("goals", "sum")
        )
        .sort_values(["top10_appearances", "top10_goals"], ascending=[False, False])
)

player_top10_count

,player,top10_appearances,top10_goals
0,Alan Shearer,10,236
78,Harry Kane,9,210
127,Mohamed Salah,8,184
171,Thierry Henry,7,164
164,Sergio Agüero,7,152
...,...,...,...
168,Steve Claridge,1,12
1,Alan Smith,1,11
66,Frédéric Kanouté,1,11
75,Gustavo Poyet,1,11


In [41]:
season_summary = (
    data.groupby("season", as_index=False)
         .agg(
             top10_total_goals=("goals", "sum"),
             top10_avg_goals=("goals", "mean"),
             top10_player_count=("player", "count")
         )
         .sort_values("top10_total_goals", ascending=False)
)


season_summary


,season,top10_total_goals,top10_avg_goals,top10_player_count
1,1993/94,261,21.75000,12
2,1994/95,247,20.58333,12
26,2018/19,247,16.46667,15
24,2016/17,230,19.16667,12
25,2017/18,227,17.46154,13
0,1992/93,215,16.53846,13
17,2009/10,215,19.54545,11
30,2022/23,213,19.36364,11
31,2023/24,210,19.09091,11
4,1996/97,207,15.92308,13


In [44]:
highest_scoring_season = season_summary.iloc[0]
lowest_scoring_season = season_summary.iloc[-1]

print(f"Highest scoring season: {highest_scoring_season['season']} with {highest_scoring_season['top10_total_goals']} goals")
print(f"Lowest scoring season: {lowest_scoring_season['season']} with {lowest_scoring_season['top10_total_goals']} goals")      

Highest scoring season: 1993/94 with 261 goals
Lowest scoring season: 2006/07 with 145 goals


In [45]:
club_summary = (
    data.groupby("club", as_index=False)
         .agg(
             total_top10_goals=("goals", "sum"),
             top10_appearances=("player", "count"),
             unique_players=("player", "nunique")
         )
         .sort_values(["total_top10_goals", "top10_appearances"], ascending=[False, False])
)

club_summary

,club,total_top10_goals,top10_appearances,unique_players
22,Manchester United,741,41,21
20,Liverpool,700,37,13
0,Arsenal,675,37,16
21,Manchester City,600,32,17
35,Tottenham Hotspur,569,31,14
11,Chelsea,418,25,12
24,Newcastle United,338,17,9
1,Aston Villa,260,17,10
4,Blackburn Rovers,224,11,7
32,Southampton,210,13,8


In [46]:
nationality_summary = (
    top10.groupby("nationality", as_index=False)
         .agg(
             total_top10_goals=("goals", "sum"),
             top10_appearances=("player", "count"),
             unique_players=("player", "nunique")
         )
         .sort_values(["total_top10_goals", "top10_appearances"], ascending=[False, False])
)

print("\nNationalities with the most top 10 goals / appearances:")
print(nationality_summary.head(20).to_string(index=False))


Nationalities with the most top 10 goals / appearances:
        nationality  total_top10_goals  top10_appearances  unique_players
            England               2890                163              68
             France                511                 30              12
        Netherlands                334                 18               6
             Norway                219                 11               7
            Belgium                211                 13               4
          Argentina                209                 10               2
              Egypt                184                  8               1
              Spain                156                  9               6
             Brazil                134                  9               9
           Portugal                117                  6               3
            Nigeria                116                  8               3
            Senegal                112                 

In [ ]:
nationality_players = (
    data.sort_values(["nationality", "goals"], ascending=[True, False])
        .groupby("nationality", as_index=False)
        .agg(
            total_top10_goals=("goals", "sum"),
            top10_appearances=("player", "count"),
            unique_players=("player", "nunique"),
            players=("player", lambda x: ", ".join(x))
        )
        .sort_values(["total_top10_goals", "top10_appearances"], ascending=[False, False])
)


print(nationality_players.head(20).to_string(index=False))

           nationality                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  

In [48]:
for _, row in nationality_players.iterrows():
    print(f"\n{row['nationality']} ({row['total_top10_goals']} goals)")
    print(row["players"])


England (2890 goals)
Andy Cole, Alan Shearer, Alan Shearer, Alan Shearer, Kevin Phillips, Harry Kane, Harry Kane, Harry Kane, Robbie Fowler, Wayne Rooney, Wayne Rooney, Matt Le Tissier, Chris Sutton, Robbie Fowler, Les Ferdinand, Alan Shearer, Harry Kane, Les Ferdinand, Darren Bent, Jamie Vardy, Ian Wright, Ian Wright, Alan Shearer, Alan Shearer, James Beattie, Jamie Vardy, Harry Kane, Teddy Sheringham, Stan Collymore, Alan Shearer, Frank Lampard, Danny Ings, Cole Palmer, Andy Cole, Andy Johnson, Daniel Sturridge, Harry Kane, Les Ferdinand, Peter Beardsley, Jamie Vardy, Raheem Sterling, Ivan Toney, Dean Holdsworth, Mark Bright, Matt Le Tissier, Michael Bridges, Andy Cole, Marcus Stewart, Michael Owen, Michael Owen, Phil Foden, Dominic Solanke, Ollie Watkins, Teddy Sheringham, Ian Wright, Robbie Fowler, Dion Dublin, Michael Owen, Chris Sutton, Michael Owen, Darren Bent, Jermain Defoe, Charlie Austin, Dele Alli, Raheem Sterling, Jamie Vardy, Harry Kane, Callum Wilson, Micky Quinn, Dean 

In [50]:
with pd.ExcelWriter("premier_league_summaries.xlsx", engine="openpyxl") as writer:
    player_goals.to_excel(writer, sheet_name="player_goals", index=False)
    player_top10_count.to_excel(writer, sheet_name="top10_counts", index=False)
    season_summary.to_excel(writer, sheet_name="season_summary", index=False)
    club_summary.to_excel(writer, sheet_name="club_summary", index=False)
    nationality_summary.to_excel(writer, sheet_name="nationality_summary", index=False)